In [ ]:
import sys
sys.path.append('..')  # Go up one level
import matching_decoupling_network as network
import numpy as np

# A Generator of Three-Element Matching (Impedance Transform) Networks

## Notations
M, N correspond to T-shaped networks.

O, P correspond to Π-shaped networks.

Component numbering:
Coil ← 1 2 3 → Amplifier

<img src="../assets/jn_matching_networks_3.png" alt="Three-Element Network" width="200" />

Need: 
- Frequency in hertz
- $Z_\mathrm{amp}$: in ohm
- $Z_\mathrm{out}$: in ohm
- $Z_\mathrm{coil}$: in ohm

The quantities above are defined in the picture below. 

<img src="../assets/preamplifier_decoupling_configuration.png" alt="Preamplifier Decoupling Configuration" width="600" />


## Choosing between Cases
Cases can be "HZ" for high impedance, "LZ" for low impedance: 
- To suppress the loop current of a loop MRI coil (i.e. an electrically small loop antenna), use "HZ". In this case:
    - Set $Z_\mathrm{coil}$ as the coil impedance. A rough coil reactance and a rough estimation of the coil resistance will usually make a decent initial design. 
    - Set $Z_\mathrm{out}$ as the optimal noise impedance of the preamplifier module. 
    - Set $Z_\mathrm{amp}$ as the input impedance of the preamplifier module.
    - $Z_\mathrm{in}$ will be automatically calculated.
- To make low-impedance amplifiers, use "LZ". In this case: 
    - Set $Z_\mathrm{coil}$ as the target optimal noise impedance, e.g. $Z_\mathrm{coil}=50~\Omega$, or anything you need.
    - Set $Z_\mathrm{out}$ as the optimal noise impedance of the "true" amplifier, i.e. the optimal noise impedance seen at the input of the very first transistor stage.
    - Set $Z_\mathrm{amp}$ as the input impedance of the "true" amplifier, i.e. the input impedance seen at the input of the very first transistor stage.
    - $Z_\mathrm{in}$ will be automatically calculated.
- To form preamp decoupling for parallel self-resonant coil loops, use "LZ". In this case:
    - Set $Z_\mathrm{coil}$ as the coil impedance. A rough coil reactance and a rough estimation of the coil resistance will usually make a decent initial design. 
    - Set $Z_\mathrm{out}$ as the optimal noise impedance of the preamplifier module.
    - Set $Z_\mathrm{amp}$ as the input impedance of the preamplifier module.
    - $Z_\mathrm{in}$ will be automatically calculated.


## Example 1: Making a Low-Input-Impedance Preamplifier

Here we show an example to make a low-input-impedance amplifier. It is made of an Infineon BFP740 BJT. The circuit is shown in Figure 4 of [1], repeated below. We would like to make the optimal noise impedance to be 50 Ω when seeing from the input of matching network (Port 1 in the picture above).

<img src="../assets/amplifier_bfp740.jpeg" alt="Amplifier based on BFP740" width="500" />

The parameters are in Table 1 of \[1\], also repeated below. 

```math
\begin{aligned}
f &= 127.732\times 10^6~\mathrm{Hz} \\
Z_\mathrm{amp} &= 551\angle -47.0^\circ~\Omega \\
Z_\mathrm{out} &= 45.5\angle 0.143^\circ~\Omega \\
Z_\mathrm{coil} &= 50~\Omega
\end{aligned}
```
We choose case `LZ` for this purpose.

\[1\] W. Wang, V. Zhurbenko, J. D. Sánchez-Heredia, and J. H. Ardenkjær-Larsen. “Trade-off between preamplifier noise figure and decoupling in MRI detectors”. In: Magnetic Resonance in Medicine 89 (2 Feb. 2023), pp. 859–871. ISSN: 15222594. DOI: 10.1002/mrm.29489. [Open Access](https://onlinelibrary.wiley.com/doi/full/10.1002/mrm.29489). 

↓---The following parameters can be changed---↓

In [ ]:
freq = 127.732e6   # Frequency, Hz
z_amp = 551. * np.exp(1j * np.deg2rad(-47.0))  # Amplifier input impedance, Ohm
z_out = 45.5 * np.exp(1j * np.deg2rad(0.143))  # The impedance to transfer the coil Z to, Ohm. It's often the noise optimum.
z_coil = 50.  # Coil impedance, Ohm
category = "LZ"   # Can be "HZ" or "LZ"

↑---The above parameters can be changed---↑

### Example 1: Results
M, N correspond to T-shaped networks.

O, P correspond to Π-shaped networks.

Component numbering:
Coil ← 1 2 3 → Amplifier

In [ ]:
assert freq > 0, "Frequency must be positive"
assert category in ["HZ", "LZ"], "Category must be \"HZ\" or \"LZ\""

x11, xø_p, x22 = 0, 0, 0
match category:
    case "HZ":
        x11, xø_p, x22 = network.impedance_matrix_hz(z_coil, z_amp, z_out)
        z_pr = network.power_swr(z_out, z_amp) * np.real(z_coil) - 1j * np.imag(z_coil)
    case "LZ":
        x11, xø_p, x22 = network.impedance_matrix_lz(z_coil, z_amp, z_out)
        z_pr = np.real(z_coil) / network.power_swr(z_out, z_amp) - 1j * np.imag(z_coil) 
decoupling = network.max_preamplifier_decoupling(z_out, z_amp)
relative_gain = network.gain_relative_to_maximum_available(z_out, z_amp)
xM1, xM2, xM3 = network.reactance_matrix_to_T(x11, xø_p, x22)
xN1, xN2, xN3 = network.reactance_matrix_to_T(x11, -xø_p, x22)
bO1, bO2, bO3 = network.reactance_matrix_to_Pi(x11, xø_p, x22)
bP1, bP2, bP3 = network.reactance_matrix_to_Pi(x11, -xø_p, x22)
M1_to_N3 = network.reactance_to_LC(np.array([xM1, xM2, xM3, xN1, xN2, xN3]), freq)
O1_to_P3 = network.susceptance_to_LC(np.array([bO1, bO2, bO3, bP1, bP2, bP3]), freq)

print(f"Maximum preamplifier decoupling = {decoupling: .5f} = {-20 * np.log10(decoupling): .2f} dB")
print(f"Gain relative to the maximum available = {relative_gain: .5f} = {10 * np.log10(relative_gain): .2f} dB")
print(f"Z_in = {z_pr: .6g} Ω")
print(f"""X_11 = {x11: .6g} Ω
X_⌀ = ±{xø_p: .6g} Ω
X_22 = {x22: .6g} Ω
""")
print(f"""
M1={M1_to_N3[0][0]: .6g} {M1_to_N3[1][0]}\t M2={M1_to_N3[0][1]: .6g} {M1_to_N3[1][1]}\t M3={M1_to_N3[0][2]: .6g} {M1_to_N3[1][2]}
N1={M1_to_N3[0][3]: .6g} {M1_to_N3[1][3]}\t N2={M1_to_N3[0][4]: .6g} {M1_to_N3[1][4]}\t N3={M1_to_N3[0][5]: .6g} {M1_to_N3[1][5]}
O1={O1_to_P3[0][0]: .6g} {O1_to_P3[1][0]}\t O2={O1_to_P3[0][1]: .6g} {O1_to_P3[1][1]}\t O3={O1_to_P3[0][2]: .6g} {O1_to_P3[1][2]}
P1={O1_to_P3[0][3]: .6g} {O1_to_P3[1][3]}\t P2={O1_to_P3[0][4]: .6g} {O1_to_P3[1][4]}\t P3={O1_to_P3[0][5]: .6g} {O1_to_P3[1][5]}
""")

## Example 2: Making a Three-Element Matching Network for a Loop MRI Coil

We still use the amplifier shown in Figure 4 of \[1\]. Now we would like to construct a matching (impedance transform) network to present a high impedance to the loop coil. The data are
```math
\begin{aligned}
f &= 127.732\times 10^6~\mathrm{Hz} \\
Z_\mathrm{amp} &= 551\angle -47.0^\circ~\Omega \\
Z_\mathrm{out} &= 45.5\angle 0.143^\circ~\Omega \\
Z_\mathrm{coil} &= 0.536 - \mathrm{j} 285~\Omega
\end{aligned}
```
We choose case `HZ` for this purpose.

↓---The following parameters can be changed---↓

In [ ]:
freq = 127.732e6   # Frequency, Hz
z_amp = 551. * np.exp(1j * np.deg2rad(-47.0))  # Amplifier input impedance, Ohm
z_out = 45.5 * np.exp(1j * np.deg2rad(0.143))  # The impedance to transfer the coil Z to, Ohm. It's often the noise optimum.
z_coil = 0.536 - 285j  # Coil impedance, Ohm
category = "HZ"   # Can be "HZ" or "LZ"

↑---The above parameters can be changed---↑

### Example 2: Results
M, N correspond to T-shaped networks.

O, P correspond to Π-shaped networks.

Component numbering:
Coil ← 1 2 3 → Amplifier

The numbers should match those in Table 1 of \[1\], namely,
```math
\begin{aligned}
X_{11} &= 294~\Omega \\
X_\varnothing &= \pm 81.7~\Omega \\
X_{22} &= 751~\Omega
\end{aligned}
```

In [ ]:
assert freq > 0, "Frequency must be positive"
assert category in ["HZ", "LZ"], "Category must be \"HZ\" or \"LZ\""

x11, xø_p, x22 = 0, 0, 0
match category:
    case "HZ":
        x11, xø_p, x22 = network.impedance_matrix_hz(z_coil, z_amp, z_out)
        z_pr = network.power_swr(z_out, z_amp) * np.real(z_coil) - 1j * np.imag(z_coil)
    case "LZ":
        x11, xø_p, x22 = network.impedance_matrix_lz(z_coil, z_amp, z_out)
        z_pr = np.real(z_coil) / network.power_swr(z_out, z_amp) - 1j * np.imag(z_coil) 
decoupling = network.max_preamplifier_decoupling(z_out, z_amp)
relative_gain = network.gain_relative_to_maximum_available(z_out, z_amp)
xM1, xM2, xM3 = network.reactance_matrix_to_T(x11, xø_p, x22)
xN1, xN2, xN3 = network.reactance_matrix_to_T(x11, -xø_p, x22)
bO1, bO2, bO3 = network.reactance_matrix_to_Pi(x11, xø_p, x22)
bP1, bP2, bP3 = network.reactance_matrix_to_Pi(x11, -xø_p, x22)
M1_to_N3 = network.reactance_to_LC(np.array([xM1, xM2, xM3, xN1, xN2, xN3]), freq)
O1_to_P3 = network.susceptance_to_LC(np.array([bO1, bO2, bO3, bP1, bP2, bP3]), freq)

print(f"Maximum preamplifier decoupling = {decoupling: .5f} = {-20 * np.log10(decoupling): .2f} dB")
print(f"Gain relative to the maximum available = {relative_gain: .5f} = {10 * np.log10(relative_gain): .2f} dB")
print(f"Z_in = {z_pr: .6g} Ω")
print(f"""X_11 = {x11: .6g} Ω
X_⌀ = ±{xø_p: .6g} Ω
X_22 = {x22: .6g} Ω
""")
print(f"""
M1={M1_to_N3[0][0]: .6g} {M1_to_N3[1][0]}\t M2={M1_to_N3[0][1]: .6g} {M1_to_N3[1][1]}\t M3={M1_to_N3[0][2]: .6g} {M1_to_N3[1][2]}
N1={M1_to_N3[0][3]: .6g} {M1_to_N3[1][3]}\t N2={M1_to_N3[0][4]: .6g} {M1_to_N3[1][4]}\t N3={M1_to_N3[0][5]: .6g} {M1_to_N3[1][5]}
O1={O1_to_P3[0][0]: .6g} {O1_to_P3[1][0]}\t O2={O1_to_P3[0][1]: .6g} {O1_to_P3[1][1]}\t O3={O1_to_P3[0][2]: .6g} {O1_to_P3[1][2]}
P1={O1_to_P3[0][3]: .6g} {O1_to_P3[1][3]}\t P2={O1_to_P3[0][4]: .6g} {O1_to_P3[1][4]}\t P3={O1_to_P3[0][5]: .6g} {O1_to_P3[1][5]}
""")